In [1]:

import os
import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table, QTable, join, unique
from matplotlib import pyplot as plt
from multiprocessing import Pool, cpu_count
from tqdm.notebook import tqdm
# vstack is used to concatenate tables
from astropy.table import vstack

In [2]:
alma_regions = Table.read('../alma_input/almaprops_2024_all_20250219.tab',format='ascii.ipac')
alma_sources = Table.from_pandas(
    pd.read_csv(
        "../alma_input/7MTM2TM1_Core_catalogue_out_official_v3_run6Apr23+16Jan24_sn-5_may24_clump_cat.txt",
        comment="\\",
        skiprows=[2],
        delimiter="\s+",
        )
    )

In [3]:
spitzer_sources= Table.read('../output/tables/spitzer_sources_glimpse_s07.tbl',format='ascii.ipac')
match_table = Table.read('../output/tables/match_table_glimpse_s07.tbl',format='ascii.ipac')
print(len(spitzer_sources))
print(len(match_table))

102073
589


In [4]:
print(len(match_table))
print(len(unique(match_table, keys=['region_id','alma_source'])))

print(len(alma_sources))
print(len(unique(alma_sources, keys=['CLUMP','ID'])))

589
589
6348
6348


In [5]:
print(alma_sources['CLUMP'].dtype)
print(match_table['region_id'].dtype)

print(alma_sources['ID'].dtype)
print(match_table['alma_source'].dtype)

<U18
<U18
int64
int64


In [6]:
print(alma_sources[['CLUMP','ID']][:5])
print(match_table[['region_id','alma_source']][:5])

CLUMP   ID
------ ---
100132   1
100132   2
100244   4
100309   1
100309  12
    region_id     alma_source
----------------- -----------
            47184          14
            49005          12
G010.8856+00.1221           5
            50555          13
            51554           4


In [30]:
matched = join(
    alma_sources,
    match_table,
    keys_left=['CLUMP', 'ID'],
    keys_right=['region_id', 'alma_source'],
    join_type= 'inner'
)

#matched.remove_column('alma_ra_01')

print(len(matched))
print(len(match_table))
print(matched.colnames)
matched.remove_columns(['alma_ra','alma_dec','region_id','alma_source'])


cols = list(matched.colnames)

cols_move = ['spitzer_source','spitzer_ra','spitzer_dec','separation_arcsec']
for i,col_element in enumerate(cols_move):  
    cols.remove(col_element)
    cols.insert(7+i,col_element)
print(cols)
matched = matched[cols]

print(matched.colnames)

589
589
['ID', 'CLUMP', 'X', 'Y', 'RA', 'DEC', 'GLON', 'GLAT', 'F_PEAK', 'S/N', 'FWHM_X', 'FWHM_Y', 'FWHM_max/FWHM_min', 'FWHM_circ', 'Bmaj/Bmin', 'BEAM_circ', 'PA', 'F_INT', 'RMS_map', 'CLUMP.1', 'N_FRAG', 'DIST', 'Lclump', 'Mclump', 'Tclump', 'Surfd', 'Tcore', 'Mcore', 'Dcore', 'BEAM_size', 'Dcore/BEAM_size', 'Lclump/Mclump', 'EVOL_FLAG', 'Dclump', 'Subsample', 'GAUSS_FLAG', 'Dcore_min', 'Dcore_max', 'Bmaj', 'Bmin', 'BPA', 'Back_onpeak', 'ID_ord', 'F_PEAK_jybm', 'Back_onpeak_jybm', 'Rcore', 'Surfd_core', 'n(H2)', 'AG_ID', 'ORIG_ID', 'RADIO_INFO', 'RADIO_MATCH', 'Surfd_nd', 'VLSR_clump', 'region_id', 'alma_source', 'spitzer_source', 'alma_ra', 'alma_dec', 'spitzer_ra', 'spitzer_dec', 'separation_arcsec']
['ID', 'CLUMP', 'X', 'Y', 'RA', 'DEC', 'GLON', 'spitzer_source', 'spitzer_ra', 'spitzer_dec', 'separation_arcsec', 'GLAT', 'F_PEAK', 'S/N', 'FWHM_X', 'FWHM_Y', 'FWHM_max/FWHM_min', 'FWHM_circ', 'Bmaj/Bmin', 'BEAM_circ', 'PA', 'F_INT', 'RMS_map', 'CLUMP.1', 'N_FRAG', 'DIST', 'Lclump', 

In [8]:
if np.isin(matched.colnames,['X']):
    print('yay')

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
match_keys = set(zip(match_table["region_id"], match_table["alma_source"]))

alma_sources["has_spitzer"] = [
    (r, s) in match_keys
    for r, s in zip(alma_sources["CLUMP"], alma_sources["ID"])
]

In [40]:
spitzer_sources['has_ALMA'] = [i in (match_table['spitzer_source']) for i in spitzer_sources['designation']]

In [43]:
print(len(spitzer_sources[spitzer_sources['has_ALMA'] == True]))
print(len(spitzer_sources[spitzer_sources['has_ALMA'] == False]))

518
101555
